In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [4]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-Gear");

In [5]:
BoSSSshell.WorkflowMgm.Init("EE-Gear");

Project name is set to 'EE-Gear'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\EE-Gear'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'C:\Users\miao\Documents\Database\EE-Gear'.


## Create grid

In [7]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid(int h) { 
        double xLeft = -1;
        double xRight = 1;
        double yTop = 1;
        double yBottom = -1;
        int kelem = Convert.ToInt32(Math.Pow(2, h));;
        
        double[] CutOut1Point1 = new double[2] {  0.05, -0.05 }; 
        double[] CutOut1Point2 = new double[2] {  -0.05, 0.05 };
        
        var CutOut1 = new BoSSS.Platform.Utils.Geom.BoundingBox(2);
        CutOut1.AddPoint(CutOut1Point1);
        CutOut1.AddPoint(CutOut1Point2);

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, CutOuts: CutOut1);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        //grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
        grd.EdgeTagNames.Add(3, "wall_left");
        grd.EdgeTagNames.Add(4, "wall_right");
        grd.EdgeTagNames.Add(5, "wall_inside");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 5;

            if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [39]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocityX(double[] X) {");
           stw.WriteLine("    double A = 1.047120418848;");
           stw.WriteLine("    double B = -0.094240837696;");
           stw.WriteLine("    double v_x = -(A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double InletVelocityY(double[] X) {");
           stw.WriteLine("    double A = 1.047120418848;");
           stw.WriteLine("    double B = -0.094240837696;");
           stw.WriteLine("    double v_y = (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementX(double[] X, double t) {");
           stw.WriteLine("    double A = 1.047120418848;");
           stw.WriteLine("    double B = -0.094240837696;");
           stw.WriteLine("    double dis_x = - t * (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementY(double[] X, double t) {");
           stw.WriteLine("    double A = 1.047120418848;");
           stw.WriteLine("    double B = -0.094240837696;");
           stw.WriteLine("    double dis_y = t * (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double Pressure(double[] X) {");
           stw.WriteLine("    double A = 1.047120418848;");
           stw.WriteLine("    double B = -0.094240837696;");
           stw.WriteLine("    double rho = 1.0;");
           stw.WriteLine("    double p0 = 1;");
           stw.WriteLine("    double r = Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           //stw.WriteLine("    double p = p0 + rho * (0.5 * A * A * r * r + 2 * A * B * Math.Log(r) - 0.5 * (B * B) / (r * r));");
           stw.WriteLine("    return 0.046 / 0.3;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           //stw.WriteLine("    return -(X[1] * X[1] + X[0] * X[0] - 0.36);");
           stw.WriteLine("    }"); 
           
//           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
//           stw.WriteLine("    return -(X[1] * X[1] / 0.25 + X[0] * X[0] - 0.09);");
//           stw.WriteLine("    }"); 
            
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    double h = 0.1;");
           stw.WriteLine("    if(X[0] >= (-0.3) && X[0] < (0.3)) {");
           stw.WriteLine("        return -(X[1]).Pow2() + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    if(X[0] < (-0.3)) {");
           stw.WriteLine("    return -((X[1]).Pow2() + (X[0] - (-0.3)).Pow2()) + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    return -((X[1]).Pow2() + (X[0] - ( 0.3)).Pow2()) + h * h;");
           stw.WriteLine("    }"); 
            
            
           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocityX(){
        return new Formula("BoundaryValues.InletVelocityX", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InletVelocityY(){
        return new Formula("BoundaryValues.InletVelocityY", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementX(){
        return new Formula("BoundaryValues.PreDisplacementX", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementY(){
        return new Formula("BoundaryValues.PreDisplacementY", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_Pressure(){
        return new Formula("BoundaryValues.Pressure", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [40]:
public static ZLS_Control Couette( int p = 2, int h = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Gear";
    C.SessionName = "6Gear-Taylor-Couette-p" + p + "-h" + h;
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.rho_B = 2.0;
    C.PhysicalParameters.mu_A = 0.05;
    C.PhysicalParameters.mu_B = 0.1;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 1000,
        //Viscosity = 0.1
        Lame2 = 1e1,
        Viscosity = 0.05
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid(h));
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddInitialValue("VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddInitialValue("VelocityX#C", BoundaryValueFactory.Get_InletVelocityX());
    C.AddInitialValue("VelocityY#C", BoundaryValueFactory.Get_InletVelocityY());
    C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Pressure#C", BoundaryValueFactory.Get_Pressure());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_lower", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_upper", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_upper", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    //C.AddBoundaryValue("pressure_outlet_upper");
    C.AddBoundaryValue("wall_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_left", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_right", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_right", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_inside", "VelocityX#A", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_inside", "VelocityY#A", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_inside", "VelocityX#C", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_inside", "VelocityY#C", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_inside", "DisplacementX#C", BoundaryValueFactory.Get_PreDisplacementX());
    C.AddBoundaryValue("wall_inside", "DisplacementY#C", BoundaryValueFactory.Get_PreDisplacementY());
    //C.AddBoundaryValue("wall_inside");



    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    C.PhysicalParameters.sliplength = 0.0;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = false;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = false;
    //C.DynamicLoadBalancing_Period = 500;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    double dt = 1e-2;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 1000;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [41]:
double[] Density = new double[] {1}; 
int[] Resolution = new int[] {5}; 
int[] DgOrder = new int[] {3}; 

In [42]:
foreach(int DgO in DgOrder){
foreach(int Res in Resolution){
foreach(double Den in Density){
    var C_CTRLFILE = Couette(DgO, Res, 2, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 2;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput(); 
}
}
}



Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job 6Gear-Taylor-Couette-p3-h5 ... 
Set Database: { Session Count = 23; Grid Count = 25; Path = C:\Users\miao\Documents\Database\EE-Gear }
Grid is not in database yet...
Grid successfully saved: 57885c18-ad37-40c5-862d-c4060ae6125d


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\Deployment\EE-Gear-ZwoLevelSetSolver2024Aug11_090220.618841
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [25]:
wmg.Sessions

#0: EE-Gear	3Gear-Taylor-Couette-p3-h5*	08/11/2024 08:40:01	17da26e1...
#1: EE-Gear	2Gear-Taylor-Couette-p3-h5*	08/11/2024 08:37:01	4bd5cc07...
#2: EE-Gear	1Gear-Taylor-Couette-p3-h5*	08/11/2024 08:36:01	2ed97f0b...
#3: EE-Gear	Gear-Taylor-Couette-p3-h5*	08/11/2024 08:31:21	7f405dca...
#4: EE-Gear	Gear16-Taylor-Couette-p3-h5*	08/10/2024 15:46:09	c26be126...
#5: EE-Gear	Gear15-Taylor-Couette-p3-h5*	08/10/2024 15:44:19	e4424978...
#6: EE-Gear	Gear14-Taylor-Couette-p3-h5*	08/10/2024 15:41:19	b08586f5...
#7: EE-Gear	Gear13-Taylor-Couette-p3-h5*	08/10/2024 15:36:59	3b4acf60...
#8: EE-Gear	Gear12-Taylor-Couette-p3-h5*	08/10/2024 15:31:49	1694ce82...
#9: EE-Gear	Gear11-Taylor-Couette-p3-h5*	08/10/2024 15:28:30	55fd5b33...
#10: EE-Gear	Gear10-Taylor-Couette-p3-h5*	08/10/2024 14:41:49	fc560883...
#11: EE-Gear	Gear9-Taylor-Couette-p3-h5*	08/10/2024 14:38:39	993252e9...
#12: EE-Gear	Gear8-Taylor-Couette-p3-h5*	08/10/2024 14:32:39	d029e7ee...
#13: EE-Gear	Gear7-Taylor-Couette-p3-h5*	08/10/2024 14:

In [37]:
wmg.Sessions[0].Timesteps.Count

3

In [45]:
var outPath = wmg.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-Gear__6Gear-Taylor-Couette-p3-h5__1f81a0d1-50db-477b-8736-2d637318fca8


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [142]:
wmg.Sessions

#0: EE-FS-Couette-Flow	2Fluid-Solid-Couette-p2-h6	08/08/2024 15:39:22	5d916d08...
#1: EE-FS-Couette-Flow	2Fluid-Solid-Couette-p2-h5	08/08/2024 15:37:24	75340a97...
#2: EE-FS-Couette-Flow	2Fluid-Solid-Couette-p2-h4	08/08/2024 15:37:24	1fc91999...
#3: EE-FS-Couette-Flow	2Fluid-Solid-Couette-p2-h3	08/08/2024 15:37:13	017f6fe3...
#4: EE-FS-Couette-Flow	2Fluid-Solid-Couette-p2-h2	08/08/2024 15:37:13	7c902480...


In [143]:
var session = wmg.Sessions[0];

In [144]:
session.Timesteps.Count

51

In [145]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("DisplacementX").ProbeAt(0.0, 0.25),
        t => "DisplacementX"
        );

In [146]:
mydataset.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.0001 
 
 
 
 
 0.0002 
 
 
 
 
 0.0003 
 
 
 
 
 0.0004 
 
 
 
 
 0.0005 
 
 
 
 
 0.0006 
 
 
 
 
 0.0007 
 
 
 
 
 0.0008 
 
 
 
 
 0.0009 
 
 
 
 
 0 
 
 
 
 
 10 
 
 
 
 
 20 
 
 
 
 
 30 
 
 
 
 
 40 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DisplacementX 
 
 
 DisplacementX 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M533.0,57.1 L586.4,57.1 M78.8,564.0 L92.2,557.0 L105.7,540.8 L119.1,508.9 L132.5,460.6 L145.9,402.5
 L159.4,344.0 L172.8,291.6 L186.2,248.0 L199.6,212.9 L213.1,184.9 L226.5,162.8 L239.9,145.2 L253.3,131.2
 L266.8,120.1 L280.2,111.1 L293.6,104.0 L307.0,98.3 L320.5,93.7 L333.9,90.1 L347.3,87.2 L360.7,84.8
 L374.2,83.0 L387.6,81.5 L401.0,80.3 L414.4,79.3 L427.9,78.6 L441.3,78.0 L454.7,77.5 L468.2,77.1
 L481.6,76.8 L495.0,76.5 L508.4,76.3 L521.9,76.2 L535.3,76.0 L548.7,75.9 L562.1,75.9 L575.6,75.8
 L589.0,75.7 L602.4,75.7 L615.8,75.7 L629.3,75.6 L642.7,75.6 L656.1,75.6 L669.5,75.6 L683.0,75.6
 L696.4,75.6 L709.8,75.6 L723.2,75.6 L736.7,75.5 L750.1,75.5 '/>

In [149]:
for (int i = 0; i < 5; i++) 
{
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  //Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

0.0008327539119221423
0.000832753905586246
0.0008327538562921613
0.0008327534738945147
0.0008327505542550956


In [132]:
int[] Cases = new int[] {0, 1, 3, 4}; 

In [133]:
foreach(int i in Cases){
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

0.000832753905586246
0.3337694306042147
0.0008327538562921613
0.3337694745333356
0.0008327534738945147
0.33376982641370684
0.0008327505542550956
0.3337725817949061


In [50]:
var DispX = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();

In [51]:
DispX.ProbeAt(0.0, 0.25)

0.0008327781162742156

In [53]:
Displacement


(1,1): error CS0103: The name 'Displacement' does not exist in the current context



Error: compilation error

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].